#### Tagger connection and setup

In [1]:
import TimeTagger
import itertools
import numpy as np
from TimeTagger import TimeTaggerBase, Coincidence, Coincidences, CoincidenceTimestamp, FileReader, TimeTagStream, Correlation, Counter, DelayedChannel
import time
import csv 

tagger = TimeTagger.createTimeTagger()
#tagger.setHardwareBufferSize(536870912) # was 67108864(536870912)

#frequency_channel = 1 
#PPS_channel = 2 
# Define the hardware settings here, such as trigger level or dead time. 
for ch in [1, 2, 3, 4]:
    tagger.setTriggerLevel(ch, -0.3)

tagger.setInputDelay(-1, 0) #ps
tagger.setInputDelay(-2, 0)
tagger.setInputDelay(-3, 6200)
tagger.setInputDelay(-4, 6200)

ch = [-1, -2, -3, -4]
bw = 500_000        # trial = 500 ns [ps]

In [2]:
def measure_patterns(T=30):

    T_ps = int(T * 1e12)
    N_trial = T_ps // bw

    pattern_counts = np.zeros(16, dtype=np.int64)

    last_trial = None
    last_mask = 0

    def process(data):
        nonlocal last_trial, last_mask

        ts = np.asarray(data.getTimestamps())
        cs = np.asarray(data.getChannels())

        if len(ts) == 0:
            return

        trial = (ts - data.tStart) // bw

        # -1 -> 0001, -2 -> 0010, -3 -> 0100, -4 -> 1000
        bits = 1 << (-cs - 1)

        good = (trial >= 0) & (trial < N_trial)
        trial = trial[good]
        bits = bits[good]

        if len(trial) == 0:
            return

        starts = np.r_[0, np.where(np.diff(trial) != 0)[0] + 1]
        tr = trial[starts]
        masks = np.bitwise_or.reduceat(bits, starts)

        for ti, mask in zip(tr, masks):

            if last_trial is None:
                last_trial = ti
                last_mask = mask

            elif ti == last_trial:
                last_mask |= mask

            else:
                pattern_counts[int(last_mask)] += 1
                last_trial = ti
                last_mask = mask

    stream = TimeTagger.TimeTagStream(
        tagger,
        n_max_events=1_000_000,
        channels=ch
    )

    stream.stop()
    stream.startFor(T_ps, clear=True)

    total_tags = 0

    while stream.isRunning():
        data = stream.getData()
        total_tags += data.size
        process(data)
        time.sleep(0.05)

    data = stream.getData()
    total_tags += data.size
    process(data)

    if last_trial is not None:
        pattern_counts[int(last_mask)] += 1

    # 0000
    pattern_counts[0] = N_trial - pattern_counts[1:].sum()

    return pattern_counts, N_trial, total_tags

#### Function generator setup

In [2]:
### Turn off the pulse source
import pyvisa

rm = pyvisa.ResourceManager()
print(rm.list_resources())
# Bob
awg = rm.open_resource("USB0::0x0957::0x5707::MY53801707::INSTR")
# Alice
#awg = rm.open_resource("USB0::0x0957::0x5707::MY53802358::INSTR")
awg.timeout = 5000

awg.write("*CLS")
awg.write("SOUR1:FUNC PULS")
awg.write("SOUR1:FREQ 10E6")
awg.write("SOUR1:FUNC:PULS:WIDT 5E-9")
awg.write("SOUR1:VOLT 0.001")      # 1 Vpp
awg.write("SOUR1:VOLT:OFFS 0") # 0 V offset
awg.write("UNIT:ANGL SEC")
awg.write("SOUR1:PHAS 20E-9")   # ns      # phase in ns
awg.write("OUTP1 OFF")

print(awg.query("SYST:ERR?"))

('USB0::0x0957::0x1745::SERIALxxxx::INSTR', 'ASRL1::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL6::INSTR', 'USB0::0x0957::0x5707::MY53801707::0::INSTR', 'USB0::0x0957::0x5707::MY53802358::0::INSTR')


VisaIOError: VI_ERROR_NCIC (-1073807264): The interface associated with this session is not currently the controller in charge.

In [70]:
rate = TimeTagger.Countrate(tagger, ch)
time.sleep(1)

rates = rate.getData()
for c, r in zip(ch, rates):
    print(f"CH{abs(c)}: {r:.0f} counts/s")

CH1: 38462 counts/s
CH2: 37645 counts/s
CH3: 38455 counts/s
CH4: 35318 counts/s


In [ ]:
voltage_list = []
counts_list = []
Ntrial_list = []

# 1. Dark measurement
awg.write("SOUR1:VOLT 0.001")
awg.write("OUTP1 OFF")
time.sleep(2)
print("Signal off")
print("Start taking dark measurement...")
counts, N_trial, total_tags = measure_patterns(T=10)

voltage_list.append(0.0)
counts_list.append(counts)
Ntrial_list.append(N_trial)

print("\nDark:")
print("Total tags =", total_tags)

for mask in range(16):
    print(f"{mask:04b}: {counts[mask]}")

# 2. Coherent-state sweep

awg.write("OUTP1 ON")
print("Start sweeping ...")

n_power = 5
voltages = np.linspace(0.5, 2.5, n_power)

for k in range(n_power):

    # set power 
    V = voltages[k]
    awg.write(f"SOUR1:VOLT {V}")
    print(f"Point {k+1}/{n_power}: V = {V:.2f} V")
    time.sleep(1)

    # Measure 16 click patterns
    counts, N_trial, total_tags = measure_patterns(T=10)

    voltage_list.append(V)
    counts_list.append(counts)
    Ntrial_list.append(N_trial)

    #print("Total tags =", total_tags)
    print("Pattern counts =", counts)

voltage_arr = np.array(voltage_list)
counts_arr = np.array(counts_list)
Ntrial_arr = np.array(Ntrial_list)
pattern_prob = counts_arr / Ntrial_arr[:, None]


Signal off
Start taking dark measurement...

Dark:
Total tags = 1449797
0000: 18605429
0001: 342265
0010: 349271
0011: 6451
0100: 343902
0101: 6385
0110: 6530
0111: 116
1000: 321321
1001: 5902
1010: 6216
1011: 102
1100: 5871
1101: 117
1110: 121
1111: 1
Start sweeping ...
Point 1/5: V = 0.50 V
Pattern counts = [15416707  1040344  1008743    86105  1028388    88094    86206    11307
   957404    81347    80024    10487    81557    10664    10591     2032]
Point 2/5: V = 1.00 V
Pattern counts = [9868242 1964240 1861185  386515 1923276  400314  378324   85151 1787753
  370788  352970   78567  364038   81517   77477   19643]
Point 3/5: V = 1.50 V
Pattern counts = [4993043 2153545 2019320  887624 2099578  923725  865829  389376 1938878
  853316  800504  358700  832448  372947  349656  161511]
Point 4/5: V = 2.00 V
Pattern counts = [2107487 1624061 1511803 1243728 1586010 1302405 1211347 1002524 1447462
 1187147 1104397  914301 1158426  960026  892118  746758]
Point 5/5: V = 2.50 V
Pattern co

In [ ]:
print("voltage_arr =", voltage_arr)
print("counts_arr =", counts_arr)
print("Ntrial_arr =", Ntrial_arr)
print("pattern_prob =", pattern_prob)

In [73]:
np.savez(
    "four_pixel_sweep_data.npz",
    voltage_arr=voltage_arr,
    counts_arr=counts_arr,
    Ntrial_arr=Ntrial_arr,
    pattern_prob=pattern_prob
)

In [6]:

TimeTagger.freeTimeTagger(tagger)

#### Fitting and reconstruction